# Exploratory Data Analysis (EDA) — Branch Dataset

This notebook explores a branch directory with columns such as `sl_no`, `state`, `branch_name`, and `branch_address`.

**Before running:** place your CSV or Excel file in the notebook's working directory and update `DATA_PATH` in the next cell. The notebook supports `.csv`, `.xlsx`, and `.xls` files.


In [ ]:
# Cell 1: Imports and configuration
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_PATH = "branches.xlsx"  # Change to your file name, e.g. "branches.csv"


In [ ]:
# Cell 2: Load the dataset
path = Path(DATA_PATH)

if not path.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH!r}. Put the file in the notebook's working directory "
        "or update DATA_PATH to the correct path."
    )

suffix = path.suffix.lower()
if suffix == ".csv":
    df = pd.read_csv(path)
elif suffix in [".xlsx", ".xls"]:
    df = pd.read_excel(path)
else:
    raise ValueError("Unsupported file type. Use CSV, XLSX, or XLS.")

# Standardize column names: trim whitespace and convert to lowercase snake_case
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(f"Loaded: {path.name}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())


In [ ]:
# Cell 3: Initial structure and summary
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes.rename("dtype").to_frame())

print("\\nInfo:")
df.info()

print("\\nDescriptive statistics (all columns):")
display(df.describe(include="all").T)


In [ ]:
# Cell 4: Missing values
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "non_missing_count": df.notna().sum()
}).sort_values("missing_percent", ascending=False)

display(missing)

ax = missing["missing_percent"].plot(kind="bar", figsize=(10, 4), title="Missing values by column (%)")
ax.set_ylabel("Missing (%)")
ax.set_xlabel("Column")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 5: Duplicate records
exact_duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows: {exact_duplicate_count:,} ({exact_duplicate_count / max(len(df), 1) * 100:.2f}%)")

if exact_duplicate_count:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(20))

# Check likely business-key duplicates, if those columns exist
for col in ["sl_no", "branch_name", "branch_address"]:
    if col in df.columns:
        print(f"Duplicate non-null values in {col!r}: {int(df[col].dropna().duplicated().sum()):,}")


In [ ]:
# Cell 6: Clean text columns for analysis (does not overwrite the original data)
text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
df_eda = df.copy()

for col in text_cols:
    df_eda[col] = df_eda[col].astype("string").str.strip()
    # Treat empty strings as missing in the analysis copy
    df_eda[col] = df_eda[col].replace(r"^\\s*$", pd.NA, regex=True)

print("Text columns:", text_cols)
print("Blank/whitespace-only values after trimming:")
display(df_eda[text_cols].isna().sum().rename("blank_or_missing_count").to_frame())


In [ ]:
# Cell 7: Categorical distributions
categorical_cols = df_eda.select_dtypes(include=["object", "string", "category"]).columns.tolist()

for col in categorical_cols:
    print(f"\\n{col}: {df_eda[col].nunique(dropna=True)} unique non-null values")
    display(df_eda[col].value_counts(dropna=False).head(15).rename_axis(col).to_frame("count"))

# State distribution is especially useful for a branch directory
state_col = next((c for c in df_eda.columns if c in ["state", "state_name"]), None)
if state_col:
    state_counts = df_eda[state_col].value_counts().head(20).sort_values()
    plt.figure(figsize=(10, max(4, len(state_counts) * 0.32)))
    state_counts.plot(kind="barh")
    plt.title("Number of branches by state (top 20)")
    plt.xlabel("Branch count")
    plt.ylabel("State")
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 8: Numeric analysis
numeric_cols = df_eda.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_cols)

if numeric_cols:
    display(df_eda[numeric_cols].describe().T)
    for col in numeric_cols:
        plt.figure(figsize=(8, 3))
        sns.histplot(df_eda[col].dropna(), kde=True)
        plt.title(f"Distribution of {col}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric columns were detected.")


In [ ]:
# Cell 9: Branch-name and address quality checks
branch_col = next((c for c in df_eda.columns if c in ["branch_name", "branch", "branch_nm", "branch_ne"]), None)
address_col = next((c for c in df_eda.columns if "address" in c), None)

if branch_col:
    print("Most common branch names:")
    display(df_eda[branch_col].value_counts(dropna=False).head(20).to_frame("count"))
    print("Branch-name values with leading/trailing whitespace in original data:")
    original = df[branch_col].astype("string")
    stripped = original.str.strip()
    display(df.loc[original.notna() & (original != stripped), [branch_col]].head(20))

if address_col:
    address = df_eda[address_col].astype("string")
    address_lengths = address.str.len()
    print("Address length summary:")
    display(address_lengths.describe().to_frame("address_character_count"))

    plt.figure(figsize=(9, 4))
    sns.histplot(address_lengths.dropna(), bins=30)
    plt.title("Distribution of address lengths")
    plt.xlabel("Characters in address")
    plt.tight_layout()
    plt.show()

    print("Shortest addresses:")
    display(df_eda.loc[address_lengths.nsmallest(10).index, [address_col]])
    print("Longest addresses:")
    display(df_eda.loc[address_lengths.nlargest(10).index, [address_col]])


In [ ]:
# Cell 10: Extract possible Indian PIN codes from addresses
# This finds standalone 6-digit sequences; manually validate results because OCR/source data may contain errors.
if address_col:
    df_eda["extracted_pin"] = (
        df_eda[address_col]
        .astype("string")
        .str.extract(r"(?<!\\d)(\\d{6})(?!\\d)", expand=False)
    )
    print("Rows with a detected 6-digit PIN:", int(df_eda["extracted_pin"].notna().sum()))
    print("Rows without a detected 6-digit PIN:", int(df_eda["extracted_pin"].isna().sum()))
    display(df_eda["extracted_pin"].value_counts().head(20).to_frame("count"))

    if state_col:
        print("PIN extraction coverage by state:")
        pin_coverage = df_eda.groupby(state_col, dropna=False)["extracted_pin"].agg(
            total_rows="size",
            pin_found="count"
        )
        pin_coverage["coverage_percent"] = (
            pin_coverage["pin_found"] / pin_coverage["total_rows"] * 100
        ).round(2)
        display(pin_coverage.sort_values("coverage_percent"))
else:
    print("No address-like column was found, so PIN extraction was skipped.")


In [ ]:
# Cell 11: Potential data-quality issues
issues = {}

# Missing values
issues["rows_with_any_missing_value"] = int(df_eda.isna().any(axis=1).sum())

# Duplicate rows
issues["exact_duplicate_rows"] = int(df_eda.duplicated().sum())

# Duplicate serial numbers, if present
if "sl_no" in df_eda.columns:
    issues["duplicate_sl_no_values"] = int(df_eda["sl_no"].dropna().duplicated().sum())

# Suspiciously short addresses
if address_col:
    issues["addresses_under_15_characters"] = int(
        df_eda[address_col].dropna().astype("string").str.len().lt(15).sum()
    )

# Missing key fields when available
for key in [state_col, branch_col, address_col]:
    if key:
        issues[f"missing_{key}"] = int(df_eda[key].isna().sum())

display(pd.Series(issues, name="count").to_frame())


In [ ]:
# Cell 12: Optional state-by-branch summary
if state_col and branch_col:
    state_branch_summary = (
        df_eda.groupby(state_col, dropna=False)
        .agg(
            branch_records=(branch_col, "size"),
            unique_branch_names=(branch_col, "nunique")
        )
        .sort_values("branch_records", ascending=False)
    )
    display(state_branch_summary)
else:
    print("Could not identify both state and branch-name columns.")


In [ ]:
# Cell 13: Export cleaned analysis copy and data-quality report
# This only trims text and converts blank strings to missing values; it does not correct spelling or infer missing data.
output_data = "branches_eda_cleaned.xlsx"
output_report = "branches_data_quality_report.xlsx"

df_eda.to_excel(output_data, index=False)
with pd.ExcelWriter(output_report, engine="openpyxl") as writer:
    missing.to_excel(writer, sheet_name="Missing values")
    pd.Series(issues, name="count").to_frame().to_excel(writer, sheet_name="Quality checks")
    if state_col and "state_branch_summary" in globals():
        state_branch_summary.to_excel(writer, sheet_name="State summary")

print(f"Saved analysis copy to: {output_data}")
print(f"Saved data-quality report to: {output_report}")
